# IJIES EL-Fleet Reproducibility Notebook

This notebook reproduces the **retained** statistical and machine-learning workflow for the 204-panel EL fleet study.

Included:
- dataset integrity checks;
- fleet descriptive statistics;
- Pearson and Spearman association;
- 80/20 hold-out evaluation;
- 5-fold cross-validation;
- Linear Regression and Random Forest;
- Composite Health Score (PPI/CSI/TRI; nominal 40/35/25 weights);
- ±5 percentage-point Health-Score weight sensitivity.

Intentionally excluded:
- bootstrap confidence intervals;
- repeated cross-validation;
- temporal validation;
- morphology ablation;
- competing top-20 maintenance-ranking baselines;
- dynamic PV energy modelling, XFEM, Charles–Hillig, and XGBoost.

**Scope:** this notebook reproduces the retained panel-level statistical and machine-learning workflow described for the 204-panel fleet study.


In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
from scipy.stats import pearsonr, spearmanr
from sklearn.model_selection import train_test_split, KFold, cross_val_score
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import r2_score, mean_absolute_error

RANDOM_STATE = 42
DATA_FILE = Path("IJIES_EL_PV_Fleet_204_Panels.csv")

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 160)


## 1. Load and verify the 204-panel analytical dataset

In [ ]:
df = pd.read_csv(DATA_FILE)

required = [
    "ID_PV", "Rated_Power", "Actual_Power", "Power_Loss",
    "Crack_Area", "HotSpot_Area", "Edge_Damage", "Num_Cracks",
    "Severity_Level"
]
missing = [c for c in required if c not in df.columns]
if missing:
    raise ValueError(f"Missing required columns: {missing}")

print("Shape:", df.shape)
print("Duplicate ID_PV:", int(df["ID_PV"].duplicated().sum()))
print("Missing values in required columns:", int(df[required].isna().sum().sum()))

assert len(df) == 204, "Expected 204 panel-level records."
assert df["ID_PV"].duplicated().sum() == 0, "Duplicate panel identifiers found."
assert df[required].isna().sum().sum() == 0, "Missing values found in required analytical fields."

df.head()


## 2. Fleet descriptive statistics

In [ ]:
fleet_summary = pd.Series({
    "N_panels": len(df),
    "Rated_capacity_kWp": df["Rated_Power"].sum() / 1000,
    "Mean_power_loss_pct": df["Power_Loss"].mean(),
    "Mean_actual_power_W": df["Actual_Power"].mean(),
    "Min_power_loss_pct": df["Power_Loss"].min(),
    "Max_power_loss_pct": df["Power_Loss"].max(),
})
display(fleet_summary.to_frame("Value"))

severity = df["Severity_Level"].value_counts().reindex(
    ["Good", "Minor", "Moderate", "Severe", "Critical", "Catastrophic"]
).fillna(0).astype(int)
display(severity.to_frame("Panels"))


## 3. Association analysis

In [ ]:
predictors = ["Num_Cracks", "Edge_Damage", "HotSpot_Area", "Crack_Area"]
association_rows = []

for col in predictors:
    pr, pp = pearsonr(df[col], df["Power_Loss"])
    sr, sp = spearmanr(df[col], df["Power_Loss"])
    association_rows.append({
        "Feature": col,
        "Pearson_r": pr,
        "Pearson_p": pp,
        "Spearman_rho": sr,
        "Spearman_p": sp,
    })

association = pd.DataFrame(association_rows)
display(association)


## 4. Outcome-independent predictive comparison

In [ ]:
feature_sets = {
    "Crack_count_only": ["Num_Cracks"],
    "Crack_area_only": ["Crack_Area"],
    "EL_composite": ["Num_Cracks", "Crack_Area", "HotSpot_Area", "Edge_Damage"],
}

models = {
    "Linear Regression": LinearRegression(),
    "Random Forest": RandomForestRegressor(
        n_estimators=300,
        max_depth=5,
        random_state=RANDOM_STATE
    ),
}

y = df["Power_Loss"].to_numpy()
kf = KFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
prediction_rows = []

for feature_name, cols in feature_sets.items():
    X = df[cols].to_numpy()
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.20, random_state=RANDOM_STATE
    )

    for model_name, model in models.items():
        model.fit(X_train, y_train)
        y_pred = model.predict(X_test)

        cv_r2 = cross_val_score(model, X, y, cv=kf, scoring="r2")
        cv_mae = -cross_val_score(
            model, X, y, cv=kf, scoring="neg_mean_absolute_error"
        )

        prediction_rows.append({
            "Feature_Set": feature_name,
            "Model": model_name,
            "HoldOut_R2": r2_score(y_test, y_pred),
            "HoldOut_MAE": mean_absolute_error(y_test, y_pred),
            "CV5_R2_mean": cv_r2.mean(),
            "CV5_R2_std": cv_r2.std(),
            "CV5_MAE_mean": cv_mae.mean(),
        })

prediction_results = pd.DataFrame(prediction_rows)
display(prediction_results)


## 5. Composite Health Score

The score is retained as a composite condition / maintenance-prioritisation index.

- PPI = `1 - Power_Loss/100`
- CSI = inverse min-max normalization of `Crack_Area`
- TRI = inverse min-max normalization of `HotSpot_Area`
- nominal weights = 0.40 / 0.35 / 0.25
- edge-damage penalty = 5 points

The Health Score contains Power_Loss through PPI and is therefore **not treated as an outcome-independent predictor of Power_Loss**.


In [ ]:
def compute_health_score(data, w_ppi=0.40, w_csi=0.35, w_tri=0.25, edge_penalty=0.05):
    ppi = 1.0 - data["Power_Loss"] / 100.0

    crack_range = data["Crack_Area"].max() - data["Crack_Area"].min()
    hotspot_range = data["HotSpot_Area"].max() - data["HotSpot_Area"].min()

    if crack_range == 0 or hotspot_range == 0:
        raise ValueError("Cannot min-max normalize a constant Crack_Area or HotSpot_Area column.")

    csi = 1.0 - (
        (data["Crack_Area"] - data["Crack_Area"].min()) / crack_range
    )
    tri = 1.0 - (
        (data["HotSpot_Area"] - data["HotSpot_Area"].min()) / hotspot_range
    )

    score = (
        (w_ppi * ppi + w_csi * csi + w_tri * tri) * 100
        - data["Edge_Damage"] * edge_penalty * 100
    )
    return score.clip(0, 100)

df["Health_Score"] = compute_health_score(df)

def health_band(score):
    if score >= 80:
        return "Healthy"
    if score >= 60:
        return "Degraded"
    if score >= 40:
        return "At Risk"
    return "Critical"

df["Health_Category"] = df["Health_Score"].map(health_band)

print("Mean Health Score:", df["Health_Score"].mean())
display(df["Health_Category"].value_counts().reindex(
    ["Healthy", "Degraded", "At Risk", "Critical"]
).fillna(0).astype(int).to_frame("Panels"))


## 6. Health-Score weight sensitivity

In [ ]:
scenarios = {
    "Nominal": (0.40, 0.35, 0.25),
    "S1": (0.45, 0.30, 0.25),
    "S2": (0.35, 0.40, 0.25),
    "S3": (0.40, 0.30, 0.30),
    "S4": (0.35, 0.35, 0.30),
}

TOP_N = 20
nominal_score = compute_health_score(df, *scenarios["Nominal"])
nominal_rank = nominal_score.rank(ascending=True)
nominal_worst = set(df.loc[nominal_score.nsmallest(TOP_N).index, "ID_PV"])

sensitivity_rows = []
for name, weights in scenarios.items():
    score = compute_health_score(df, *weights)
    rank = score.rank(ascending=True)
    rho, p_value = spearmanr(nominal_rank, rank)
    scenario_worst = set(df.loc[score.nsmallest(TOP_N).index, "ID_PV"])

    sensitivity_rows.append({
        "Scenario": name,
        "w_PPI": weights[0],
        "w_CSI": weights[1],
        "w_TRI": weights[2],
        "Spearman_rho_vs_Nominal": rho,
        "Top20_Worst_Overlap_pct": 100 * len(nominal_worst & scenario_worst) / TOP_N,
    })

sensitivity_results = pd.DataFrame(sensitivity_rows)
display(sensitivity_results)


## 7. Univariate association versus 5-fold predictive performance

In [ ]:
univariate_rows = []

for col in predictors:
    X = df[[col]].to_numpy()
    pr, pp = pearsonr(df[col], y)
    sr, sp = spearmanr(df[col], y)

    model = LinearRegression()
    cv_r2 = cross_val_score(model, X, y, cv=kf, scoring="r2")
    cv_mae = -cross_val_score(
        model, X, y, cv=kf, scoring="neg_mean_absolute_error"
    )

    univariate_rows.append({
        "Predictor": col,
        "Pearson_r": pr,
        "Pearson_p": pp,
        "Spearman_rho": sr,
        "Spearman_p": sp,
        "CV5_R2": cv_r2.mean(),
        "CV5_MAE": cv_mae.mean(),
    })

univariate_results = pd.DataFrame(univariate_rows)
display(univariate_results)


## 8. Export reproducibility tables

These files contain only the retained analyses in the final study scope.


In [ ]:
OUTPUT_DIR = Path("reproducibility_outputs")
OUTPUT_DIR.mkdir(exist_ok=True)

association.to_csv(OUTPUT_DIR / "association_results.csv", index=False)
prediction_results.to_csv(OUTPUT_DIR / "predictive_results.csv", index=False)
sensitivity_results.to_csv(OUTPUT_DIR / "health_score_weight_sensitivity.csv", index=False)
univariate_results.to_csv(OUTPUT_DIR / "association_vs_prediction.csv", index=False)

print("Saved reproducibility tables to:", OUTPUT_DIR.resolve())
